# 3. Evaluación multiedición (requiere GPU)

Corre CLIP-B, CLIP-L y LongCLIP sobre las tres ediciones del IESF, evaluando
**cada edición como un pool de recuperación independiente**.

Esto convierte el experimento de una sola medición (n=40) en un diseño de
**réplicas independientes**: si el orden de los modelos se repite en 2021, 2024 y
2026, la conclusión deja de depender de un documento particular.

> Requiere descargar los tres checkpoints desde Hugging Face y GPU con ~8 GB.
> Los notebooks 1 y 2 no necesitan nada de esto.

In [1]:
from pathlib import Path
import json
import sys

import numpy as np
import pandas as pd
import torch
from PIL import Image

BASE = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
MULTI = BASE / "multiedicion"

# --- variante de imágenes -------------------------------------------------
# Un único punto decide si se trabaja con las imágenes que llevan el título
# impreso o con las recortadas. El manifiesto es el mismo para ambas; cambia
# solo la subcarpeta. El nombre del directorio de salida arrastra la variante,
# de modo que dos corridas nunca se sobrescriben.
sys.path.insert(0, str(BASE))
from config_variante import (VARIANTE, carpeta_imagenes, ruta_imagen,
                             dir_salida, resumen)

IMGS = carpeta_imagenes(MULTI)
OUT = dir_salida(BASE, "resultados_multiedicion")
print(resumen(MULTI))
print(f"salida:  {OUT}")

MODEL_SPECS = {
    "CLIP":     {"checkpoint": "openai/clip-vit-base-patch32",
                 "max_tokens": 77,  "encoder_visual": "ViT-B/32"},
    "CLIP-L":   {"checkpoint": "openai/clip-vit-large-patch14",
                 "max_tokens": 77,  "encoder_visual": "ViT-L/14"},
    "LongCLIP": {"checkpoint": "zer0int/LongCLIP-GmP-ViT-L-14",
                 "max_tokens": 248, "encoder_visual": "ViT-L/14"},
}
CAPTIONS = ["caption_2", "caption_3", "caption_4"]
KS = (1, 5, 10)

man = pd.read_csv(MULTI / "manifest_multiedicion.csv")
print(f"{len(man)} pares, {man['edicion'].nunique()} ediciones")
print(man.groupby("edicion").size().to_string())
print("\nGPU disponible:", torch.cuda.is_available())

variante: con_titulo | carpeta: con_titulo | 95 imágenes
salida:  /tf/work/final/sbs_iesf_pares/resultados_multiedicion_con_titulo
95 pares, 3 ediciones
edicion
2021-1    26
2024-2    29
2026-1    40

GPU disponible: True


## 3.1 Carga de modelos y embeddings

Los tres modelos usan la misma interfaz de `transformers`. LongCLIP comparte la
arquitectura de CLIP-L con los embeddings posicionales extendidos a 248 tokens.

Si el checkpoint de LongCLIP no carga con `CLIPModel.from_pretrained`, hay que
reemplazar esta función por la que usa el Cuaderno14 original: es la única parte
que puede necesitar ajuste.

In [2]:
from transformers import CLIPModel, CLIPProcessor, CLIPConfig

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
BATCH = 8


def cargar_modelo(nombre):
    """Carga modelo y procesador.

    use_safetensors=True evita la ruta de torch.load, que transformers bloquea
    con torch anterior a 2.6 (CVE-2025-32434).

    Para LongCLIP hay que ampliar max_position_embeddings a 248 ANTES de cargar
    los pesos: si no, el modelo se instancia con las 77 posiciones de CLIP y la
    ventana larga —la variable central del estudio— nunca se usa.
    """
    spec = MODEL_SPECS[nombre]
    ck, maxlen = spec["checkpoint"], spec["max_tokens"]

    if maxlen > 77:
        cfg = CLIPConfig.from_pretrained(ck)
        cfg.text_config.max_position_embeddings = maxlen
        modelo = CLIPModel.from_pretrained(ck, config=cfg, use_safetensors=True)
    else:
        modelo = CLIPModel.from_pretrained(ck, use_safetensors=True)

    proc = CLIPProcessor.from_pretrained(ck)
    modelo = modelo.to(DEVICE).eval()

    real = modelo.config.text_config.max_position_embeddings
    assert real == maxlen, f"{nombre}: se esperaban {maxlen} posiciones, hay {real}"
    print(f"  {nombre}: {real} posiciones de texto")
    return modelo, proc


@torch.no_grad()
def embeddings(modelo, proc, rutas, textos, max_tokens, batch=BATCH):
    """Embeddings normalizados de imágenes y textos, procesados por lotes.

    padding=True rellena solo hasta el texto más largo del lote, no hasta el
    límite del modelo. La normalización L2 es lo que convierte el producto punto
    en similitud coseno.
    """
    im = []
    for s in range(0, len(rutas), batch):
        imgs = [Image.open(pth).convert("RGB") for pth in rutas[s:s + batch]]
        inp = proc(images=imgs, return_tensors="pt").to(DEVICE)
        out = modelo.get_image_features(**inp)
        im.append(torch.nn.functional.normalize(out, dim=-1).cpu())

    tx = []
    for s in range(0, len(textos), batch):
        inp = proc(text=list(textos[s:s + batch]), return_tensors="pt",
                   padding=True, truncation=True, max_length=max_tokens).to(DEVICE)
        out = modelo.get_text_features(**inp)
        tx.append(torch.nn.functional.normalize(out, dim=-1).cpu())

    return torch.cat(im).numpy(), torch.cat(tx).numpy()

/tf/work/torch_gpu_env/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 3.2 Métricas del pool

El par correcto está en la diagonal de la matriz de similitud. `rangos` devuelve en
qué posición quedó cada uno.

El **baseline** desplaza los captions una posición: si el modelo siguiera
acertando, significaría que la métrica captura un artefacto de la matriz y no
alineamiento real. Debe dar cerca de 0.

In [3]:
def rangos(sim):
    n = sim.shape[0]
    orden = np.argsort(-sim, axis=1)          # descendente
    return np.array([int(np.where(orden[i] == i)[0][0]) + 1 for i in range(n)])


def metricas(r):
    m = {f"R@{k}": round(float((r <= k).mean()), 4) for k in KS}
    m["MRR"] = round(float((1.0 / r).mean()), 4)
    return m


def evaluar_pool(E_img, E_txt):
    sim = E_img @ E_txt.T
    r_i2t, r_t2i = rangos(sim), rangos(sim.T)
    out = {f"i2t_{k}": v for k, v in metricas(r_i2t).items()}
    out.update({f"t2i_{k}": v for k, v in metricas(r_t2i).items()})
    sim_b = E_img @ np.roll(E_txt, 1, axis=0).T          # captions desplazados
    out["baseline_i2t_R@1"] = round(float((rangos(sim_b) == 1).mean()), 4)
    return out, r_i2t

## 3.3 Corrida

Un pool por edición y por caption. Para una prueba rápida antes de la corrida
completa, reducir `modelos` a `["CLIP-L"]` y `captions` a `["caption_2"]`.

In [4]:
modelos = list(MODEL_SPECS)      # para una prueba rápida: ["CLIP-L"]
captions = CAPTIONS               # para una prueba rápida: ["caption_2"]

filas, por_caso = [], []

for nombre in modelos:
    print(f"\n--- {nombre} ({MODEL_SPECS[nombre]['encoder_visual']}, "
          f"{MODEL_SPECS[nombre]['max_tokens']} tokens) ---")
    modelo, proc = cargar_modelo(nombre)
    mt = MODEL_SPECS[nombre]["max_tokens"]

    for cap in captions:
        for edicion, g in man.groupby("edicion"):
            g = g.reset_index(drop=True)
            rutas = [ruta_imagen(MULTI, p) for p in g["image_path"]]
            E_i, E_t = embeddings(modelo, proc, rutas,
                                  g[cap].astype(str).tolist(), mt)
            m, r_i2t = evaluar_pool(E_i, E_t)
            filas.append({"modelo": nombre, "caption": cap, "edicion": edicion,
                          "n_pool": len(g), **m})
            print(f"  {cap} {edicion} (n={len(g):3d}): R@1={m['i2t_R@1']:.3f} "
                  f"MRR={m['i2t_MRR']:.3f} baseline={m['baseline_i2t_R@1']:.3f}")
            for j in range(len(g)):
                por_caso.append({"image_id": g.loc[j, "image_id"],
                                 "chart_uid": g.loc[j, "chart_uid"],
                                 "edicion": edicion, "modelo": nombre,
                                 "caption": cap, "rank_i2t": int(r_i2t[j]),
                                 "acierto": bool(r_i2t[j] == 1)})
    del modelo
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

res = pd.DataFrame(filas)
res.to_csv(OUT / "metricas_por_edicion.csv", index=False)
pd.DataFrame(por_caso).to_csv(OUT / "scores_por_caso_multiedicion.csv", index=False)
print(f"\n{len(res)} combinaciones evaluadas · {len(por_caso)} registros por caso")
print(f"salidas en {OUT}")


--- CLIP (ViT-B/32, 77 tokens) ---


Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.


  CLIP: 77 posiciones de texto
  caption_2 2021-1 (n= 26): R@1=0.269 MRR=0.455 baseline=0.154
  caption_2 2024-2 (n= 29): R@1=0.448 MRR=0.570 baseline=0.000
  caption_2 2026-1 (n= 40): R@1=0.450 MRR=0.598 baseline=0.000
  caption_3 2021-1 (n= 26): R@1=0.500 MRR=0.622 baseline=0.000
  caption_3 2024-2 (n= 29): R@1=0.414 MRR=0.550 baseline=0.035
  caption_3 2026-1 (n= 40): R@1=0.500 MRR=0.650 baseline=0.025
  caption_4 2021-1 (n= 26): R@1=0.154 MRR=0.243 baseline=0.038
  caption_4 2024-2 (n= 29): R@1=0.379 MRR=0.530 baseline=0.035
  caption_4 2026-1 (n= 40): R@1=0.475 MRR=0.619 baseline=0.050

--- CLIP-L (ViT-L/14, 77 tokens) ---
  CLIP-L: 77 posiciones de texto
  caption_2 2021-1 (n= 26): R@1=0.731 MRR=0.803 baseline=0.077
  caption_2 2024-2 (n= 29): R@1=0.690 MRR=0.774 baseline=0.035
  caption_2 2026-1 (n= 40): R@1=0.650 MRR=0.716 baseline=0.050
  caption_3 2021-1 (n= 26): R@1=0.615 MRR=0.763 baseline=0.038
  caption_3 2024-2 (n= 29): R@1=0.621 MRR=0.721 baseline=0.069
  caption_3 2026

## 3.4 Consistencia del ranking entre ediciones

La prueba de generalización. Si el orden de los modelos se repite en las tres
ediciones, la conclusión no depende del documento; si cambia, hay que decirlo.

In [5]:
# los captions se leen del propio resultado, para que la celda no dependa de
# variables que hayan quedado en memoria de una ejecución anterior
cons = []
for cap in sorted(res["caption"].unique()):
    for edicion, g in res[res.caption == cap].groupby("edicion"):
        orden = g.sort_values("i2t_R@1", ascending=False)["modelo"].tolist()
        cons.append({"caption": cap, "edicion": edicion,
                     "ranking": " > ".join(orden)})

cons = pd.DataFrame(cons)
cons.to_csv(OUT / "consistencia_ranking.csv", index=False)
print(cons.to_string(index=False))

print("\n¿el ranking es el mismo en las tres ediciones?")
for cap, g in cons.groupby("caption"):
    print(f"  {cap}: {'SI' if g['ranking'].nunique() == 1 else 'NO'}"
          f"  ({g['ranking'].nunique()} órdenes distintos)")

print("\nganador por combinación:")
gan = (res.sort_values('i2t_R@1', ascending=False)
          .groupby(['caption', 'edicion'])['modelo'].first().value_counts())
print(gan.to_string())

  caption edicion                  ranking
caption_2  2021-1 CLIP-L > LongCLIP > CLIP
caption_2  2024-2 CLIP-L > LongCLIP > CLIP
caption_2  2026-1 LongCLIP > CLIP-L > CLIP
caption_3  2021-1 LongCLIP > CLIP-L > CLIP
caption_3  2024-2 CLIP-L > LongCLIP > CLIP
caption_3  2026-1 LongCLIP > CLIP-L > CLIP
caption_4  2021-1 CLIP-L > CLIP > LongCLIP
caption_4  2024-2 CLIP-L > LongCLIP > CLIP
caption_4  2026-1 LongCLIP > CLIP-L > CLIP

¿el ranking es el mismo en las tres ediciones?
  caption_2: NO  (2 órdenes distintos)
  caption_3: NO  (2 órdenes distintos)
  caption_4: NO  (3 órdenes distintos)

ganador por combinación:
CLIP-L      5
LongCLIP    4


## 3.5 Resumen ponderado y prueba pareada sobre n=95

El promedio se pondera por el tamaño de cada pool. Y como cada gráfico fue
evaluado dentro de su propia edición, el vector de aciertos acumulado sirve para
repetir la prueba pareada con toda la muestra.

In [6]:
# promedio ponderado por tamaño de pool, sin depender de groupby.apply
# (su firma cambió entre versiones de pandas)
filas = []
for (mod, cap), g in res.groupby(["modelo", "caption"]):
    filas.append({
        "modelo": mod, "caption": cap,
        "n_total": int(g.n_pool.sum()),
        "i2t_R@1": round(float(np.average(g["i2t_R@1"], weights=g.n_pool)), 4),
        "i2t_MRR": round(float(np.average(g["i2t_MRR"], weights=g.n_pool)), 4),
    })
resumen = pd.DataFrame(filas)
resumen.to_csv(OUT / "resumen_multiedicion.csv", index=False)
print(resumen.to_string(index=False))

  modelo   caption  n_total  i2t_R@1  i2t_MRR
    CLIP caption_2       95   0.4000   0.5505
    CLIP caption_3       95   0.4737   0.6120
    CLIP caption_4       95   0.3579   0.4889
  CLIP-L caption_2       95   0.6842   0.7577
  CLIP-L caption_3       95   0.6316   0.7437
  CLIP-L caption_4       95   0.4842   0.5984
LongCLIP caption_2       95   0.6000   0.6918
LongCLIP caption_3       95   0.6421   0.7453
LongCLIP caption_4       95   0.4526   0.5673


In [7]:
from math import comb

def p_mcnemar(b, c):
    n = b + c
    if n == 0:
        return 1.0
    k = min(b, c)
    return min(1.0, 2 * sum(comb(n, i) for i in range(k + 1)) / 2 ** n)


pc = pd.DataFrame(por_caso)
piv = pc.pivot_table(index="image_id", columns=["modelo", "caption"],
                     values="acierto", aggfunc="first")

comparaciones = [
    (("CLIP", "caption_2"), ("CLIP-L", "caption_2"), "efecto tamaño"),
    (("CLIP-L", "caption_2"), ("LongCLIP", "caption_2"), "efecto contexto (cap corto)"),
    (("CLIP-L", "caption_4"), ("LongCLIP", "caption_4"), "efecto contexto (cap largo)"),
]

filas = []
for c1, c2, etq in comparaciones:
    if c1 not in piv.columns or c2 not in piv.columns:
        continue
    a, d = piv[c1].astype(bool), piv[c2].astype(bool)
    b_, c_ = int((a & ~d).sum()), int((~a & d).sum())
    p = p_mcnemar(b_, c_)
    filas.append({"comparacion": etq, "n": len(piv),
                  "solo_1": b_, "solo_2": c_, "discordancias": b_ + c_,
                  "p_valor": round(p, 4), "significativo_0.05": p < 0.05})

if filas:
    mc95 = pd.DataFrame(filas)
    mc95.to_csv(OUT / "mcnemar_n95.csv", index=False)
    print(mc95.to_string(index=False))
else:
    print("faltan condiciones; correr con los tres modelos y caption_2 y caption_3")

                comparacion  n  solo_1  solo_2  discordancias  p_valor  significativo_0.05
              efecto tamaño 95       5      32             37   0.0000                True
efecto contexto (cap corto) 95      17       9             26   0.1686               False
efecto contexto (cap largo) 95      16      13             29   0.7111               False


## 3.6 Qué esperar

Ampliar de 40 a 95 gráficos **estrecha los intervalos**, pero no puede convertir un
efecto nulo en uno significativo.

La proyección a partir de las discordancias observadas con n=40 anticipa que el
**efecto de tamaño pasa a ser significativo** mientras el **efecto de contexto
sigue sin serlo**. Si eso se confirma, el aporte real de la ampliación es poder
afirmar que la ausencia de efecto de contexto no se debe a falta de datos.

La salida más valiosa para el informe es `consistencia_ranking.csv`: responde si el
resultado se sostiene en documentos distintos, separados por cinco años de cambios
en el diseño de los gráficos.